# V2 — RAW generation (Colab)

**Drive must contain the `scripts/` folder** (copy from the repo: `Steganography_benchmarks_V2/scripts/`).

Stage 1: generate model replies **without post-processing** → `runs/<run>/raw_responses.jsonl`.

Models are cached on Drive. Afterwards: `Colab_Evaluate.ipynb`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Steganography_benchmarks_V2')
SCRIPTS_DIR = PROJECT_DIR / 'scripts'
MODEL_CACHE_DIR = PROJECT_DIR / 'models_cache'
RUNS_DIR = PROJECT_DIR / 'runs'

for d in (PROJECT_DIR, SCRIPTS_DIR, MODEL_CACHE_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

runner = SCRIPTS_DIR / 'notebook_runner.py'
if not runner.is_file():
    raise FileNotFoundError(
        f'Missing {runner}. Upload the scripts/ folder from the repo to Drive.'
    )

os.chdir(PROJECT_DIR)
os.environ['MODEL_CACHE_DIR'] = str(MODEL_CACHE_DIR)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print('PROJECT_DIR:', PROJECT_DIR)
print('scripts OK:', runner)
print('MODEL_CACHE_DIR:', MODEL_CACHE_DIR)

In [ ]:
!pip install -q -r scripts/requirements.txt

In [ ]:
from getpass import getpass
import os

if not os.getenv('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('HF_TOKEN: ')

In [ ]:
# --- CONFIG ---
TEST = 'humaneval'        # humaneval | perplexity | capacity | binoculars
MODEL = 'gemma'
THRESHOLD = 0.0
TOP_N = 16
MAX_NEW_TOKENS = 512
HUMANEVAL_TASKS = None
RESUME_RUN_DIR = None

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from notebook_runner import run_live

if RESUME_RUN_DIR:
    cmd = ['python', 'scripts/generate_responses.py', '--run-dir', RESUME_RUN_DIR]
else:
    cmd = [
        'python', 'scripts/generate_responses.py',
        '--test', TEST,
        '--model', MODEL,
        '--threshold', str(THRESHOLD),
        '--top-n', str(TOP_N),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--platform', 'colab',
        '--output-root', 'runs',
    ]
    if HUMANEVAL_TASKS and TEST == 'humaneval':
        cmd += ['--humaneval-tasks', HUMANEVAL_TASKS]

run_live(cmd)
